# Feature Engineering: OULAD Dataset
## Bias-Aware Early Warning System for Higher Education

This notebook transforms the OULAD data into LSTM-ready features for predicting at-risk students within the first 25% of course completion.

**Outputs:**
- `features_temporal.npy` - LSTM sequences (n_students, 10 weeks, 5 features)
- `features_static.csv` - Static features + protected attributes + target
- `feature_metadata.json` - Column mappings and encoding info

## 1. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data directories (relative to notebooks/)
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading OULAD dataset...")

Loading OULAD dataset...


In [2]:
# Load all tables
student_info = pd.read_csv(DATA_DIR / "studentInfo.csv")
student_vle = pd.read_csv(DATA_DIR / "studentVle.csv")
student_assessment = pd.read_csv(DATA_DIR / "studentAssessment.csv")
student_registration = pd.read_csv(DATA_DIR / "studentRegistration.csv")
assessments = pd.read_csv(DATA_DIR / "assessments.csv")
vle = pd.read_csv(DATA_DIR / "vle.csv")
courses = pd.read_csv(DATA_DIR / "courses.csv")

print(f"studentInfo: {len(student_info):,} rows")
print(f"studentVle: {len(student_vle):,} rows")
print(f"studentAssessment: {len(student_assessment):,} rows")
print(f"courses: {len(courses)} course-presentations")

studentInfo: 32,593 rows
studentVle: 10,655,280 rows
studentAssessment: 173,912 rows
courses: 22 course-presentations


In [3]:
# Calculate 25% early window cutoff for each course
courses['early_cutoff'] = (courses['module_presentation_length'] * 0.25).astype(int)

print("25% Early Window Cutoffs:")
print(courses[['code_module', 'code_presentation', 'module_presentation_length', 'early_cutoff']])

25% Early Window Cutoffs:
   code_module code_presentation  module_presentation_length  early_cutoff
0          AAA             2013J                         268            67
1          AAA             2014J                         269            67
2          BBB             2013J                         268            67
3          BBB             2014J                         262            65
4          BBB             2013B                         240            60
5          BBB             2014B                         234            58
6          CCC             2014J                         269            67
7          CCC             2014B                         241            60
8          DDD             2013J                         261            65
9          DDD             2014J                         262            65
10         DDD             2013B                         240            60
11         DDD             2014B                         241            60

In [4]:
# Create unique student-course key for joining
student_info['student_key'] = (student_info['code_module'] + '_' + 
                                student_info['code_presentation'] + '_' + 
                                student_info['id_student'].astype(str))

print(f"Unique student-course records: {len(student_info):,}")
print("\nTarget distribution:")
print(student_info['final_result'].value_counts())

Unique student-course records: 32,593

Target distribution:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64


## 2. Filter Data to Early Window (25%)

In [5]:
# Merge course info to get early cutoff for each VLE record
student_vle_with_cutoff = student_vle.merge(
    courses[['code_module', 'code_presentation', 'early_cutoff']],
    on=['code_module', 'code_presentation']
)

# Filter to early window only (date <= early_cutoff)
# Include pre-course activity (negative dates) as it may be predictive
early_vle = student_vle_with_cutoff[student_vle_with_cutoff['date'] <= student_vle_with_cutoff['early_cutoff']].copy()

print(f"Total VLE records: {len(student_vle):,}")
print(f"Early window VLE records: {len(early_vle):,} ({len(early_vle)/len(student_vle)*100:.1f}%)")

Total VLE records: 10,655,280
Early window VLE records: 4,664,953 (43.8%)


In [6]:
# Add activity type information
early_vle = early_vle.merge(vle[['id_site', 'activity_type']], on='id_site', how='left')

# Create student key for joining
early_vle['student_key'] = (early_vle['code_module'] + '_' + 
                            early_vle['code_presentation'] + '_' + 
                            early_vle['id_student'].astype(str))

print(f"\nEarly VLE date range: {early_vle['date'].min()} to {early_vle['date'].max()}")
print(f"Students with early VLE activity: {early_vle['student_key'].nunique():,}")


Early VLE date range: -25 to 67


Students with early VLE activity: 29,119


## 3. Temporal Features (Weekly Aggregation for LSTM)

Create weekly sequences covering weeks -1 to 9 (10 total weeks):
- Week -1: Pre-course activity (days -25 to -1)
- Week 0: Days 0-6
- Week 1-8: Days 7-62 (covers full 25% window)

Features per week:
1. `total_clicks` - Sum of all clicks
2. `active_days` - Number of unique days with activity
3. `n_sessions` - Count of VLE records
4. `activity_types` - Number of distinct activity types
5. `avg_daily_clicks` - Mean clicks per active day

In [7]:
def assign_week(date):
    """Assign a week number to each date.
    Week -1: days < 0 (pre-course)
    Week 0: days 0-6
    Week 1+: subsequent 7-day periods
    """
    if date < 0:
        return -1  # Pre-course activity
    else:
        return date // 7

# Assign week numbers
early_vle['week'] = early_vle['date'].apply(assign_week)

# Cap at week 9 (we want 10 weeks total: -1 to 8)
early_vle['week'] = early_vle['week'].clip(upper=8)

print("Week distribution:")
print(early_vle['week'].value_counts().sort_index())

Week distribution:
week
-1    688988
 0    573154
 1    495241
 2    580361
 3    463567
 4    413034
 5    348426
 6    376057
 7    363823
 8    362302
Name: count, dtype: int64


In [8]:
def compute_weekly_features(group):
    """Compute engagement features for a single week of student activity."""
    return pd.Series({
        'total_clicks': group['sum_click'].sum(),
        'active_days': group['date'].nunique(),
        'n_sessions': len(group),
        'activity_types': group['activity_type'].nunique(),
        'avg_daily_clicks': group['sum_click'].sum() / max(group['date'].nunique(), 1)
    })

# Aggregate by student and week
print("Computing weekly features (this may take a minute)...")
weekly_features = early_vle.groupby(['student_key', 'week']).apply(compute_weekly_features).reset_index()
print(f"Weekly feature records: {len(weekly_features):,}")

Computing weekly features (this may take a minute)...


Weekly feature records: 217,591


In [9]:
# Get all student keys from student_info (including those with no VLE activity)
all_student_keys = student_info['student_key'].unique()
n_students = len(all_student_keys)
n_weeks = 10  # Weeks -1 to 8
n_features = 5

print(f"Total students: {n_students:,}")
print(f"Sequence shape: ({n_students}, {n_weeks}, {n_features})")

Total students: 32,593
Sequence shape: (32593, 10, 5)


In [10]:
# Create mapping from student_key to index
student_key_to_idx = {key: idx for idx, key in enumerate(all_student_keys)}

# Initialize temporal array with zeros
temporal_features = np.zeros((n_students, n_weeks, n_features), dtype=np.float32)

# Feature column order
feature_cols = ['total_clicks', 'active_days', 'n_sessions', 'activity_types', 'avg_daily_clicks']

# Fill in the temporal array
print("Building temporal sequences...")
for _, row in weekly_features.iterrows():
    student_idx = student_key_to_idx.get(row['student_key'])
    if student_idx is not None:
        week_idx = int(row['week']) + 1  # Shift: week -1 -> index 0
        if 0 <= week_idx < n_weeks:
            temporal_features[student_idx, week_idx, :] = [row[col] for col in feature_cols]

print(f"Temporal array shape: {temporal_features.shape}")
print(f"Students with any activity: {(temporal_features.sum(axis=(1,2)) > 0).sum():,}")
print(f"Students with zero activity: {(temporal_features.sum(axis=(1,2)) == 0).sum():,}")

Building temporal sequences...


Temporal array shape: (32593, 10, 5)
Students with any activity: 29,119
Students with zero activity: 3,474


In [11]:
# Verify temporal features look reasonable
print("Sample temporal sequence (first student with activity):")
active_mask = temporal_features.sum(axis=(1,2)) > 0
first_active_idx = np.where(active_mask)[0][0]
print(f"Student index: {first_active_idx}")
print(f"Student key: {all_student_keys[first_active_idx]}")
print(f"\nWeekly features (rows=weeks -1 to 8, cols={feature_cols}):")
print(temporal_features[first_active_idx])

Sample temporal sequence (first student with activity):
Student index: 0
Student key: AAA_2013J_11391

Weekly features (rows=weeks -1 to 8, cols=['total_clicks', 'active_days', 'n_sessions', 'activity_types', 'avg_daily_clicks']):
[[ 98.          1.         11.          4.         98.       ]
 [183.          4.         30.          6.         45.75     ]
 [ 20.          1.          4.          3.         20.       ]
 [100.          2.         15.          5.         50.       ]
 [  0.          0.          0.          0.          0.       ]
 [ 26.          3.          8.          4.          8.666667 ]
 [ 60.          2.         12.          4.         30.       ]
 [ 22.          2.          6.          2.         11.       ]
 [ 20.          3.          4.          2.          6.6666665]
 [ 16.          1.          2.          2.         16.       ]]


## 4. Static Demographic Features

In [12]:
# Create target variable
student_info['at_risk'] = student_info['final_result'].isin(['Withdrawn', 'Fail']).astype(int)

print("Target distribution:")
print(student_info['at_risk'].value_counts())
print(f"\nAt-risk rate: {student_info['at_risk'].mean()*100:.1f}%")

Target distribution:
at_risk
1    17208
0    15385
Name: count, dtype: int64

At-risk rate: 52.8%


In [13]:
# Define encoding mappings
gender_map = {'M': 1, 'F': 0}
age_band_map = {'0-35': 0, '35-55': 1, '55<=': 2}
disability_map = {'N': 0, 'Y': 1}

education_order = [
    'No Formal quals',
    'Lower Than A Level',
    'A Level or Equivalent',
    'HE Qualification',
    'Post Graduate Qualification'
]
education_map = {ed: i for i, ed in enumerate(education_order)}

# IMD band ordering (0-10% is most deprived = highest risk)
imd_order = ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']
imd_map = {band: i for i, band in enumerate(imd_order)}

print("Encoding mappings created:")
print(f"  gender: {gender_map}")
print(f"  age_band: {age_band_map}")
print(f"  disability: {disability_map}")
print(f"  education levels: {len(education_map)}")
print(f"  imd_band levels: {len(imd_map)}")

Encoding mappings created:
  gender: {'M': 1, 'F': 0}
  age_band: {'0-35': 0, '35-55': 1, '55<=': 2}
  disability: {'N': 0, 'Y': 1}
  education levels: 5
  imd_band levels: 10


## 5. Handle Missing Data (IMD Band Imputation)

In [14]:
# Check missing values
print("Missing values in key columns:")
print(student_info[['gender', 'region', 'imd_band', 'age_band', 'disability', 'highest_education']].isnull().sum())

Missing values in key columns:
gender                  0
region                  0
imd_band             1111
age_band                0
disability              0
highest_education       0
dtype: int64


In [15]:
# Impute imd_band with mode within each region
print(f"Missing imd_band before imputation: {student_info['imd_band'].isnull().sum()}")

# Calculate mode of imd_band for each region
region_imd_mode = student_info.groupby('region')['imd_band'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else '50-60%')
print("\nIMD mode by region:")
print(region_imd_mode)

# Fill missing values
student_info['imd_band_imputed'] = student_info.apply(
    lambda row: row['imd_band'] if pd.notna(row['imd_band']) else region_imd_mode.get(row['region'], '50-60%'),
    axis=1
)

print(f"\nMissing imd_band after imputation: {student_info['imd_band_imputed'].isnull().sum()}")

Missing imd_band before imputation: 1111

IMD mode by region:
region
East Anglian Region     90-100%
East Midlands Region      10-20
Ireland                   0-10%
London Region             10-20
North Region              10-20
North Western Region      0-10%
Scotland                 50-60%
South East Region        60-70%
South Region            90-100%
South West Region        30-40%
Wales                    20-30%
West Midlands Region      0-10%
Yorkshire Region          0-10%
Name: imd_band, dtype: object



Missing imd_band after imputation: 0


In [16]:
# Apply encodings
static_features = student_info.copy()

# Encode categorical variables
static_features['gender_encoded'] = static_features['gender'].map(gender_map)
static_features['age_band_encoded'] = static_features['age_band'].map(age_band_map)
static_features['disability_encoded'] = static_features['disability'].map(disability_map)
static_features['education_encoded'] = static_features['highest_education'].map(education_map)
static_features['imd_band_encoded'] = static_features['imd_band_imputed'].map(imd_map)

# Check for any unmapped values
print("Encoded feature null counts:")
encoded_cols = ['gender_encoded', 'age_band_encoded', 'disability_encoded', 'education_encoded', 'imd_band_encoded']
print(static_features[encoded_cols].isnull().sum())

Encoded feature null counts:
gender_encoded        0
age_band_encoded      0
disability_encoded    0
education_encoded     0
imd_band_encoded      0
dtype: int64


## 6. Early Assessment Features

In [17]:
# Merge assessment info with due dates
assessment_with_dates = student_assessment.merge(
    assessments[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']],
    on='id_assessment'
)

# Merge with course info to get early cutoff
assessment_with_dates = assessment_with_dates.merge(
    courses[['code_module', 'code_presentation', 'early_cutoff']],
    on=['code_module', 'code_presentation']
)

print(f"Total assessment submissions: {len(assessment_with_dates):,}")

Total assessment submissions: 173,912


In [18]:
# Filter to assessments due within early window
early_assessments = assessment_with_dates[assessment_with_dates['date'] <= assessment_with_dates['early_cutoff']].copy()

# Create student key
early_assessments['student_key'] = (early_assessments['code_module'] + '_' + 
                                     early_assessments['code_presentation'] + '_' + 
                                     early_assessments['id_student'].astype(str))

print(f"Early window assessments: {len(early_assessments):,}")
print(f"Students with early assessments: {early_assessments['student_key'].nunique():,}")

Early window assessments: 51,394
Students with early assessments: 25,023


In [19]:
# Calculate days early (positive = submitted before deadline)
early_assessments['days_early'] = early_assessments['date'] - early_assessments['date_submitted']

# Aggregate assessment features per student
assessment_features = early_assessments.groupby('student_key').agg(
    n_assessments_submitted=('id_assessment', 'count'),
    avg_score=('score', 'mean'),
    min_score=('score', 'min'),
    max_score=('score', 'max'),
    avg_days_early=('days_early', 'mean'),
    std_score=('score', 'std')
).reset_index()

# Fill NaN std with 0 (for students with only one assessment)
assessment_features['std_score'] = assessment_features['std_score'].fillna(0)

print(f"Assessment feature records: {len(assessment_features):,}")
print(assessment_features.head())

Assessment feature records: 25,023
         student_key  n_assessments_submitted  avg_score  min_score  \
0   AAA_2013J_100893                        2       65.5       63.0   
1   AAA_2013J_101781                        2       74.0       64.0   
2   AAA_2013J_102806                        2       83.0       80.0   
3   AAA_2013J_102952                        2       73.0       70.0   
4  AAA_2013J_1035023                        2       69.0       65.0   

   max_score  avg_days_early  std_score  
0       68.0             3.0   3.535534  
1       84.0             2.5  14.142136  
2       86.0             0.0   4.242641  
3       76.0             1.0   4.242641  
4       73.0             0.0   5.656854  


In [20]:
# Count assessments due per student-course (for submission rate calculation)
# First, get unique assessments in early window per course
early_assess_due = assessments.merge(
    courses[['code_module', 'code_presentation', 'early_cutoff']],
    on=['code_module', 'code_presentation']
)
early_assess_due = early_assess_due[early_assess_due['date'] <= early_assess_due['early_cutoff']]

# Count assessments due per course
assess_due_per_course = early_assess_due.groupby(['code_module', 'code_presentation']).size().reset_index(name='n_due')

print("Assessments due in early window by course:")
print(assess_due_per_course)

Assessments due in early window by course:
   code_module code_presentation  n_due
0          AAA             2013J      2
1          AAA             2014J      2
2          BBB             2013B      3
3          BBB             2013J      3
4          BBB             2014B      3
5          BBB             2014J      2
6          CCC             2014B      2
7          CCC             2014J      3
8          DDD             2013B      4
9          DDD             2013J      2
10         DDD             2014B      2
11         DDD             2014J      3
12         EEE             2013J      1
13         EEE             2014B      1
14         EEE             2014J      1
15         FFF             2013B      2
16         FFF             2013J      2
17         FFF             2014B      2
18         FFF             2014J      2
19         GGG             2013J      1
20         GGG             2014J      1


In [21]:
# Merge assessment features with static features
static_features = static_features.merge(assessment_features, on='student_key', how='left')

# Merge assessments due count
static_features = static_features.merge(assess_due_per_course, on=['code_module', 'code_presentation'], how='left')

# Fill missing assessment features (students who submitted nothing)
static_features['n_assessments_submitted'] = static_features['n_assessments_submitted'].fillna(0)
static_features['avg_score'] = static_features['avg_score'].fillna(0)
static_features['min_score'] = static_features['min_score'].fillna(0)
static_features['max_score'] = static_features['max_score'].fillna(0)
static_features['avg_days_early'] = static_features['avg_days_early'].fillna(0)
static_features['std_score'] = static_features['std_score'].fillna(0)

# Calculate submission rate
static_features['submission_rate'] = static_features['n_assessments_submitted'] / static_features['n_due'].clip(lower=1)
static_features['submission_rate'] = static_features['submission_rate'].clip(upper=1.0)  # Cap at 100%

print(f"\nStatic features shape: {static_features.shape}")


Static features shape: (32593, 28)


## 7. Final Dataset Assembly

In [22]:
# Define final column sets
id_cols = ['student_key', 'id_student', 'code_module', 'code_presentation']

protected_cols = ['gender', 'region', 'imd_band_imputed', 'age_band', 'disability']

static_feature_cols = [
    'gender_encoded', 'age_band_encoded', 'disability_encoded', 
    'education_encoded', 'imd_band_encoded',
    'num_of_prev_attempts', 'studied_credits',
    'n_assessments_submitted', 'avg_score', 'min_score', 'max_score',
    'avg_days_early', 'std_score', 'submission_rate'
]

target_col = ['at_risk']

# Select final columns
final_static = static_features[id_cols + protected_cols + static_feature_cols + target_col].copy()

print("Final static dataset columns:")
print(f"  IDs: {id_cols}")
print(f"  Protected: {protected_cols}")
print(f"  Features: {static_feature_cols}")
print(f"  Target: {target_col}")
print(f"\nShape: {final_static.shape}")

Final static dataset columns:
  IDs: ['student_key', 'id_student', 'code_module', 'code_presentation']
  Protected: ['gender', 'region', 'imd_band_imputed', 'age_band', 'disability']
  Features: ['gender_encoded', 'age_band_encoded', 'disability_encoded', 'education_encoded', 'imd_band_encoded', 'num_of_prev_attempts', 'studied_credits', 'n_assessments_submitted', 'avg_score', 'min_score', 'max_score', 'avg_days_early', 'std_score', 'submission_rate']
  Target: ['at_risk']

Shape: (32593, 24)


In [23]:
# Verify alignment between temporal and static features
print("Verifying alignment...")

# Ensure static features are in same order as temporal array
final_static = final_static.set_index('student_key').loc[all_student_keys].reset_index()

# Verify
assert len(final_static) == len(all_student_keys), "Mismatch in record count!"
assert (final_static['student_key'].values == all_student_keys).all(), "Mismatch in order!"

print(f"Temporal features: {temporal_features.shape}")
print(f"Static features: {final_static.shape}")
print("Alignment verified!")

Verifying alignment...
Temporal features: (32593, 10, 5)
Static features: (32593, 24)
Alignment verified!


In [24]:
# Summary statistics
print("="*60)
print("FEATURE ENGINEERING SUMMARY")
print("="*60)

print("\n1. DATASET SIZE")
print(f"   Total student-course records: {len(final_static):,}")

print("\n2. TEMPORAL FEATURES (LSTM input)")
print(f"   Shape: {temporal_features.shape}")
print(f"   - {temporal_features.shape[0]:,} students")
print(f"   - {temporal_features.shape[1]} time steps (weeks)")
print(f"   - {temporal_features.shape[2]} features per time step")
print(f"   Features: {feature_cols}")

print("\n3. STATIC FEATURES")
print(f"   {len(static_feature_cols)} features")

print("\n4. PROTECTED ATTRIBUTES")
for col in protected_cols:
    n_unique = final_static[col].nunique()
    print(f"   {col}: {n_unique} categories")

print("\n5. TARGET DISTRIBUTION")
print(f"   At-Risk: {final_static['at_risk'].sum():,} ({final_static['at_risk'].mean()*100:.1f}%)")
print(f"   Not At-Risk: {(1-final_static['at_risk']).sum():,} ({(1-final_static['at_risk']).mean()*100:.1f}%)")

FEATURE ENGINEERING SUMMARY

1. DATASET SIZE
   Total student-course records: 32,593

2. TEMPORAL FEATURES (LSTM input)
   Shape: (32593, 10, 5)
   - 32,593 students
   - 10 time steps (weeks)
   - 5 features per time step
   Features: ['total_clicks', 'active_days', 'n_sessions', 'activity_types', 'avg_daily_clicks']

3. STATIC FEATURES
   14 features

4. PROTECTED ATTRIBUTES
   gender: 2 categories
   region: 13 categories
   imd_band_imputed: 10 categories
   age_band: 3 categories
   disability: 2 categories

5. TARGET DISTRIBUTION
   At-Risk: 17,208 (52.8%)
   Not At-Risk: 15,385 (47.2%)


## 8. Save Outputs

In [25]:
# Save temporal features
np.save(OUTPUT_DIR / 'features_temporal.npy', temporal_features)
print(f"Saved: {OUTPUT_DIR / 'features_temporal.npy'}")

# Save static features
final_static.to_csv(OUTPUT_DIR / 'features_static.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'features_static.csv'}")

Saved: ../data/processed/features_temporal.npy


Saved: ../data/processed/features_static.csv


In [26]:
# Save metadata
metadata = {
    'temporal_features': {
        'shape': list(temporal_features.shape),
        'feature_names': feature_cols,
        'week_mapping': {i: f'week_{i-1}' for i in range(10)},  # index 0 = week -1
        'description': 'Weekly aggregated VLE engagement features'
    },
    'static_features': {
        'id_columns': id_cols,
        'protected_columns': protected_cols,
        'feature_columns': static_feature_cols,
        'target_column': 'at_risk'
    },
    'encodings': {
        'gender': gender_map,
        'age_band': age_band_map,
        'disability': disability_map,
        'highest_education': education_map,
        'imd_band': imd_map
    },
    'preprocessing': {
        'early_window_pct': 0.25,
        'imd_imputation': 'mode_by_region',
        'n_weeks': 10,
        'week_range': '-1 to 8 (index 0 = pre-course)'
    }
}

with open(OUTPUT_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {OUTPUT_DIR / 'feature_metadata.json'}")

Saved: ../data/processed/feature_metadata.json


In [27]:
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"\nOutput files in '{OUTPUT_DIR}':")
print(f"  1. features_temporal.npy - LSTM sequences {temporal_features.shape}")
print(f"  2. features_static.csv - Static features ({len(final_static):,} rows, {len(final_static.columns)} cols)")
print("  3. feature_metadata.json - Encoding mappings and metadata")
print("\nReady for LSTM model training!")


FEATURE ENGINEERING COMPLETE

Output files in '../data/processed':
  1. features_temporal.npy - LSTM sequences (32593, 10, 5)
  2. features_static.csv - Static features (32,593 rows, 24 cols)
  3. feature_metadata.json - Encoding mappings and metadata

Ready for LSTM model training!
